In [1]:
import torch
import time
from torch import nn

### Exercise 1

Try a larger computation task, such as the multiplication of large matrices, and see the difference in speed between the CPU and GPU. What about a task with a small number of calculations?

| Version | What it measures | Correct? |
| :--- | :--- | :--- |
| No synchronize() | Launch time (enqueue) | Wrong |
| With synchronize() | Actual GPU compute time | Correct |

In [3]:
A = torch.randn(2000, 2000, device='cuda')
B = torch.randn(2000, 2000, device='cuda')

# warm-up
for _ in range(5):
    C = A @ B
torch.cuda.synchronize()

# timing without sync at the end: launch time only
torch.cuda.synchronize()
start = time.time()
C = A @ B
end = time.time()
print("Without sync:", end - start)

# timing with sync at the end: actual execution time
torch.cuda.synchronize()
start = time.time()
C = A @ B
torch.cuda.synchronize()
end = time.time()
print("With sync:", end - start)

Without sync: 6.628036499023438e-05
With sync: 0.0024039745330810547


In [4]:
def cpu():
    return torch.device('cpu')

def gpu(i=0):
    return torch.device(f'cuda:{i}')

def num_gpus():
    return torch.cuda.device_count()

def try_gpu(i=0):
    if num_gpus() >= i + 1:
        return gpu(i)
    return cpu()

def benchmark_matmul(n, device, repeats=10):
    # create random matrices on the specified device
    A = torch.randn(n, n, device=device)
    B = torch.randn(n, n, device=device)

    # warm-up
    for _ in range(3):
        C = A @ B
    if device.type == 'cuda':
        torch.cuda.synchronize()

    # timing
    start = time.time()
    for _ in range(repeats):
        C = A @ B
    if device.type == 'cuda':
        torch.cuda.synchronize()    # ensure all operations are finished before stopping the timer
    end = time.time()

    avg_time = (end - start) / repeats
    return avg_time

In [5]:
device_cpu = cpu()
device_gpu = try_gpu()

print("CPU device:", device_cpu)
print("GPU device:", device_gpu)

n = 2000

cpu_time = benchmark_matmul(n, device_cpu, repeats=10)
print(f"CPU: {n}x{n} matmul average time = {cpu_time:.6f} seconds")

if device_gpu.type == 'cuda':
    gpu_time = benchmark_matmul(n, device_gpu, repeats=10)
    print(f"GPU: {n}x{n} matmul average time = {gpu_time:.6f} seconds")
    print(f"Speedup (CPU / GPU) = {cpu_time / gpu_time:.2f}x")
else:
    print("No GPU available.")

CPU device: cpu
GPU device: cuda:0
CPU: 2000x2000 matmul average time = 0.039034 seconds
GPU: 2000x2000 matmul average time = 0.002123 seconds
Speedup (CPU / GPU) = 18.39x


In [9]:
small_sizes = [10, 20, 50, 100]

for n in small_sizes:
    cpu_time = benchmark_matmul(n, device_cpu, repeats=1000)
    print(f"\nCPU: {n}x{n} matmul average time = {cpu_time:.8f} seconds")

    if device_gpu.type == 'cuda':
        gpu_time = benchmark_matmul(n, device_gpu, repeats=1000)
        print(f"GPU: {n}x{n} matmul average time = {gpu_time:.8f} seconds")
        print(f"Speedup (CPU / GPU) = {cpu_time / gpu_time:.2f}x")


CPU: 10x10 matmul average time = 0.00000799 seconds
GPU: 10x10 matmul average time = 0.00002679 seconds
Speedup (CPU / GPU) = 0.30x

CPU: 20x20 matmul average time = 0.00003872 seconds
GPU: 20x20 matmul average time = 0.00002944 seconds
Speedup (CPU / GPU) = 1.32x

CPU: 50x50 matmul average time = 0.00006904 seconds
GPU: 50x50 matmul average time = 0.00003295 seconds
Speedup (CPU / GPU) = 2.10x

CPU: 100x100 matmul average time = 0.00008643 seconds
GPU: 100x100 matmul average time = 0.00007026 seconds
Speedup (CPU / GPU) = 1.23x


For large matrix multiplications, the GPU is significantly faster than the CPU because the operation is highly parallelizable and the GPU can exploit massive parallel hardware.
For small matrix multiplications, the GPU may offer little benefit or even be slower, because the overhead of launching GPU operations and synchronizing results can dominate the actual computation time.

### Exercise 2

How should we read and write model parameters on the GPU?

In [10]:
def try_gpu(i=0):
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

device = try_gpu()

net = nn.Sequential(
    nn.Linear(20, 10),
    nn.ReLU(),
    nn.Linear(10, 1)
).to(device)

X = torch.randn(4, 20, device=device)
y = net(X)

In [11]:
torch.save(net.state_dict(), "net.params")

In [13]:
device = try_gpu()

net2 = nn.Sequential(
    nn.Linear(20, 10),
    nn.ReLU(),
    nn.Linear(10, 1)
).to(device)

state_dict = torch.load("net.params", map_location=device, weights_only=True)
net2.load_state_dict(state_dict)

<All keys matched successfully>

In [15]:
cpu_device = torch.device('cpu')

net_cpu = nn.Sequential(
    nn.Linear(20, 10),
    nn.ReLU(),
    nn.Linear(10, 1)
)

state_dict = torch.load("net.params", map_location=cpu_device, weights_only=True)
net_cpu.load_state_dict(state_dict)

<All keys matched successfully>

In [17]:
print(net2[0].weight.device)
print(net_cpu[0].weight.device)

cuda:0
cpu


We read and write model parameters on the GPU by saving the model’s state_dict, and when loading we `use torch.load(..., map_location=desired_device)` and then `model.load_state_dict(...)` to place parameters on the correct device.

### Exercise 3

Measure the time it takes to compute 1000 matrix-matrix multiplications of $100 \times 100$ matrices and $\log$ the Frobenius norm of the output matrix one result at a time. Compare it with keeping a log on the GPU and transferring only the final result.

In [18]:
def try_gpu(i=0):
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

device = try_gpu()
print("Using device:", device)

Using device: cuda:0


In [19]:
num_iters = 1000
n = 100

A_list = [torch.randn(n, n, device=device) for _ in range(num_iters)]
B_list = [torch.randn(n, n, device=device) for _ in range(num_iters)]

In [20]:
def log_one_result_at_a_time(A_list, B_list, device):
    logs = []

    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.time()

    for A, B in zip(A_list, B_list):
        C = A @ B
        fro_norm = torch.norm(C, p='fro')
        logs.append(torch.log(fro_norm).item())   # move one scalar to CPU each time

    if device.type == 'cuda':
        torch.cuda.synchronize()
    end = time.time()

    return logs, end - start

In [21]:
def keep_log_on_gpu(A_list, B_list, device):
    logs_gpu = torch.empty(len(A_list), device=device)

    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.time()

    for i, (A, B) in enumerate(zip(A_list, B_list)):
        C = A @ B
        fro_norm = torch.norm(C, p='fro')
        logs_gpu[i] = torch.log(fro_norm)   # stay on GPU

    final_logs = logs_gpu.cpu()  # transfer once at the end

    if device.type == 'cuda':
        torch.cuda.synchronize()
    end = time.time()

    return final_logs, end - start

In [22]:
logs1, time1 = log_one_result_at_a_time(A_list, B_list, device)
logs2, time2 = keep_log_on_gpu(A_list, B_list, device)

print(f"Method 1 (transfer every time): {time1:.6f} seconds")
print(f"Method 2 (keep log on GPU):     {time2:.6f} seconds")

if time2 > 0:
    print(f"Speedup = {time1 / time2:.2f}x")

Method 1 (transfer every time): 0.147308 seconds
Method 2 (keep log on GPU):     0.065323 seconds
Speedup = 2.26x


Logging one result at a time is much slower because each log extraction triggers device-to-host transfer and synchronization overhead. Keeping the log on the GPU and transferring it only once at the end is much more efficient.

### Exercise 4

Measure how much time it takes to perform two matrix-matrix multiplications on two GPUs at the same time. Compare it with computing in sequence on one GPU. Hint: you should see almost linear scaling.

In [23]:
def gpu(i=0):
    return torch.device(f'cuda:{i}')

def num_gpus():
    return torch.cuda.device_count()

print("Number of GPUs:", num_gpus())
assert num_gpus() >= 2, "This exercise needs at least 2 GPUs."

Number of GPUs: 2


In [30]:
n = 8000

A0 = torch.randn(n, n, device=gpu(0))
B0 = torch.randn(n, n, device=gpu(0))

A1 = torch.randn(n, n, device=gpu(1))
B1 = torch.randn(n, n, device=gpu(1))

In [31]:
_ = A0 @ B0
_ = A1 @ B1
torch.cuda.synchronize(gpu(0))
torch.cuda.synchronize(gpu(1))

In [32]:
A1_seq = A1.to(gpu(0))
B1_seq = B1.to(gpu(0))
torch.cuda.synchronize(gpu(0))

In [33]:
start = time.time()

C0_seq = A0 @ B0
C1_seq = A1_seq @ B1_seq

torch.cuda.synchronize(gpu(0))
time_seq = time.time() - start

print(f"Sequential on one GPU: {time_seq:.4f} seconds")

Sequential on one GPU: 0.1209 seconds


In [34]:
start = time.time()

C0_par = A0 @ B0      # runs on cuda:0
C1_par = A1 @ B1      # runs on cuda:1

torch.cuda.synchronize(gpu(0))
torch.cuda.synchronize(gpu(1))
time_par = time.time() - start

print(f"Parallel on two GPUs:  {time_par:.4f} seconds")

Parallel on two GPUs:  0.0628 seconds


In [35]:
print(f"Speedup (sequential / parallel): {time_seq / time_par:.2f}x")

Speedup (sequential / parallel): 1.92x


Running two large matrix multiplications simultaneously on two GPUs is substantially faster than running them sequentially on a single GPU. Since each GPU can execute its own workload independently, the total runtime in the two-GPU case is close to the runtime of a single multiplication, yielding nearly linear scaling.